
### Add an MCP server to perform file operations.

MCP servers are used to facilitate file operations such as reading and writing files. They run locally and interact with the agent to provide tools for managing files. 




In [4]:
import os
from dotenv import load_dotenv, find_dotenv, dotenv_values
from agents import Agent, Runner, Tool, WebSearchTool, trace, function_tool
from agents.mcp import MCPServerStdio

# Locate .env in this directory or any parent directory
dotenv_path = find_dotenv()
if not dotenv_path:
    raise FileNotFoundError('.env not found in repository or parent directories')

# Load into os.environ (preserves existing variables unless overridden by .env)
load_dotenv(dotenv_path, override=False)
# Also read raw values as a dict (useful to expose into notebook globals)
env = {k: v for k, v in dotenv_values(dotenv_path).items() if v is not None}

# Export into notebook globals for easy access by name
globals().update(env)

print('Loaded .env from', dotenv_path)
print('Loaded keys:', list(env.keys()))

Loaded .env from /media/nathan/linux_ssd/github/agentic_ai_trip_planner/.env
Loaded keys: ['OPENAI_API_KEY', 'GROQ_API_KEY', 'PUSHOVER_USER', 'PUSHOVER_TOKEN', 'SENDGRID_API_KEY', 'GOOGLE_API_KEY', 'SERPER_API_KEY', 'LANGSMITH_TRACING', 'LANGSMITH_ENDPOINT', 'LANGSMITH_API_KEY', 'LANGSMITH_PROJECT', 'POLYGON_API_KEY', 'POLYGON_PLAN', 'BRAVE_API_KEY']



In the following code, an MCP server is configured to use a sandbox directory for file operations. The server is initialized with specific parameters, and the available file tools are listed to demonstrate its capabilities.

Note how in the previous notebook, the tool provided only had functionality to write files to disk, whereas leveraging an MCP Server provides several more file operations.

In [5]:
#Let's add an MCP server filesystem tool to read and write files.  Print out the available file tools.

sandbox_path = os.path.abspath(os.path.join(os.getcwd(), "output"))
print(f"Using sandbox path: {sandbox_path}")
filesystem_params = {"command": "npx", "args": ["-y", "@modelcontextprotocol/server-filesystem", sandbox_path]}

async with MCPServerStdio(params=filesystem_params,client_session_timeout_seconds=60) as server:
    file_tools = await server.list_tools()

file_tools

Using sandbox path: /media/nathan/linux_ssd/github/agentic_ai_trip_planner/openai_agents_sdk/output


[Tool(name='read_file', title='Read File (Deprecated)', description='Read the complete contents of a file as text. DEPRECATED: Use read_text_file instead.', inputSchema={'$schema': 'http://json-schema.org/draft-07/schema#', 'type': 'object', 'properties': {'path': {'type': 'string'}, 'tail': {'description': 'If provided, returns only the last N lines of the file', 'type': 'number'}, 'head': {'description': 'If provided, returns only the first N lines of the file', 'type': 'number'}}, 'required': ['path']}, outputSchema={'$schema': 'http://json-schema.org/draft-07/schema#', 'type': 'object', 'properties': {'content': {'type': 'string'}}, 'required': ['content'], 'additionalProperties': False}, icons=None, annotations=ToolAnnotations(title=None, readOnlyHint=True, destructiveHint=None, idempotentHint=None, openWorldHint=None), meta=None, execution=ToolExecution(taskSupport='forbidden')),
 Tool(name='read_text_file', title='Read Text File', description="Read the complete contents of a fil

### Run the agent using the MCP server

In this section, the MCP server is instantiated asynchronously to manage file operations. The server is configured with specific parameters, including a sandbox path for file operations. The `async with` statement ensures that the server is properly initialized and closed after use, preventing resource leaks.

The agent is then created with tools such as the `WebSearchTool` and the MCP server for file management. The agent uses these tools to plan a trip based on the provided directions, which include the trip duration, destination, and activities. The final trip plan is saved to a markdown file named `trip_plan`. The `trace` context manager is used to monitor the agent's execution.

In [6]:
duration = "5 days"
destination = "Paris"
activities = ["sightseeing", "dining", "museums"]
directions = f"plan a {duration} trip to {destination} including {', '.join(activities)}. Use the web search tool to find relevant information. Then save the complete trip plan to a markdown file named 'trip_plan_using_mcp.md'."

web_search_tool = WebSearchTool(search_context_size="low") #This tool can add costs to the agent

async with MCPServerStdio(params=filesystem_params, client_session_timeout_seconds=30) as mcp_server_files:
    trip_planner_agent = Agent(
        name="Trip Planner Agent",
        instructions="An agent that helps users plan trips by searching for destinations, accommodations, and activities. Use the write_file tool to save the final plan in markdown format.",
        tools=[web_search_tool],
        mcp_servers=[mcp_server_files]
    )
    with trace("Trip Planner Agent"):
        result = await Runner.run(trip_planner_agent, directions)
        print(result.final_output)

Your 5-day Paris trip plan—covering sightseeing, museums, dining, and practical tips—has been compiled based on current web research and saved as "trip_plan_using_mcp.md" in the output directory. If you need more details about any attraction or want adjustments (luxury, budget, family, etc.), just let me know!
